# FAISS vs ChromaDB

FAISS:
- In-memory vector database
- Very fast for similarity search
- Common in RAG systems
- Simple to set up

ChromaDB:
- Stores data persistently (on disk)
- Also fast and reliable
- Popular for AI applications
- Slightly heavier setup


# Complete RAG Pipeline

1. Documents
   - Load PDFs, text files, websites
   - Preprocess the content

2. Text Splitting
   - Break documents into small chunks
   - Example: chunk size = 500 tokens, overlap = 50

3. Embeddings
   - Convert every chunk into a dense vector
   - Example models: nomic-embed-text, sentence-transformers, OpenAI embeddings

4. Vector Database
   - Store vectors + metadata
   - Options: FAISS or ChromaDB

5. Retrieval
   - Convert query → embedding
   - Perform similarity search
   - Return Top-K matching chunks

6. LLM
   - Provide retrieved chunks + query to the LLM
   - LLM generates a grounded answer

7. Final Output
   - Accurate, context-aware answer
   - Based on your document data

In [11]:
from langchain_ollama import ChatOllama,OllamaEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

# doucment Loaders

In [3]:
loader = PyMuPDFLoader("/Users/yashshakya787/genAI/data/customer_support_kb.pdf")
docs = loader.load()
len(docs)

2

# text spliiter 

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)
chunks = splitter.split_documents(docs)
len(chunks)

8

# embedding

In [12]:
embedding = OllamaEmbeddings(
    model= "nomic-embed-text"
)

# create vector database

In [14]:
vector_cdb = FAISS.from_documents(
    documents=chunks,
    embedding=embedding,
)

create retriver

In [17]:
retriever = vector_cdb.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 3}
)


In [18]:
llm = ChatOllama(
    model="llama3.1:8b"
)

# create chat histroy 

In [20]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages = True
)

/var/folders/gd/yqc_8r1d50d5ldy5vyd_5lvm0000gn/T/ipykernel_53741/1682106021.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


# create chains

In [21]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever =retriever,
    memory = memory
)

# ask question 

In [22]:
query = "what is Ai"
result = qa_chain.invoke({"question": query})
result["answer"]

"I don't have any context related to AI in the provided information. However, based on general knowledge, AI stands for Artificial Intelligence, which refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.\n\nIf you're looking for more specific information about AI or would like me to provide a detailed answer, please let me know!"

In [23]:
while True:
    question = input("you: ")

    if question.lower() == "exist":
        break
    result = qa_chain.invoke({"question": question})
    print("Ai:",result["answer"])

you:  hi


Ai: Our available topics are Subscription & Plans and Refunds & Returns. Which one would you like to discuss?


you:  exist
